# ROPG-KD Data Generation
Generates `data/ropg_kd/{train,val}.jsonl` — scored (query, persona, top-K chunks) triples used to train the ROPG-KD retriever.

**Kaggle setup checklist**
1. Enable GPU accelerator (T4 × 1 is enough).
2. Enable internet access.
3. Add Kaggle secrets: `OPENAI_API_KEY` and `OPENAI_BASE_URL`.
4. Attach the `simurgh-data` dataset (contains `chunks/corpus.jsonl`, `questions/`, `splits/`).

**Colab setup checklist**
1. Set `RUNTIME = "colab"` in the Config cell below.
2. Upload `simurgh-data/` to Google Drive at `MyDrive/simurgh-data/` — must contain `chunks/corpus.jsonl`, `questions/`, `splits/`.
3. Add secrets via Colab Secrets (left sidebar → key icon): `OPENAI_API_KEY`, `OPENAI_BASE_URL`.
4. Enable GPU accelerator (T4 × 1 is enough).
5. Output is saved to `MyDrive/simurgh-data/ropg_kd/`.

In [ ]:
!pip install -q sentence-transformers openai numpy tqdm

## Config

In [ ]:
import os

# ── Runtime selector ─────────────────────────────────────────────────────────
# Set RUNTIME to match where you are running this notebook.
RUNTIME = "kaggle"  # "kaggle" | "colab" | "local"
GDRIVE_BASE = "/content/drive/MyDrive/simurgh-data"  # Colab only

# ── Secrets ───────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    from kaggle_secrets import UserSecretsClient

    _s = UserSecretsClient()
    OPENAI_API_KEY = _s.get_secret("OPENAI_API_KEY")
    OPENAI_BASE_URL = _s.get_secret("OPENAI_BASE_URL")
elif RUNTIME == "colab":
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    OPENAI_BASE_URL = userdata.get("OPENAI_BASE_URL")
else:  # local
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
    OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")

# ── Paths ─────────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    DATASET_SLUG = "simurgh-data"
    DATA_ROOT = f"/kaggle/input/datasets/alirezahsn/{DATASET_SLUG}"
    OUTPUT_DIR = "/kaggle/working/data/ropg_kd"
elif RUNTIME == "colab":
    DATA_ROOT = GDRIVE_BASE
    OUTPUT_DIR = f"{GDRIVE_BASE}/ropg_kd"
else:  # local
    DATA_ROOT = "data"
    OUTPUT_DIR = "data/ropg_kd"

# ── Inline config (mirrors configs/datagen_ropg.yaml) ────────────────────────
CFG = {
    "embedder": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "device": "cuda",  # Kaggle/Colab GPU
        "batch_size": 8,
        "fp16": True,
    },
    "retriever": {"top_k": 20},
    "judge": {
        "model": "gpt-5.4-nano",
        "temperature": 1.0,
        "max_completion_tokens": 16,
        "max_workers": 8,
    },
    "data": {
        "chunks": f"{DATA_ROOT}/chunks/corpus.jsonl",
        "questions_dir": f"{DATA_ROOT}/questions",
        "splits_dir": f"{DATA_ROOT}/splits",
        "output_dir": OUTPUT_DIR,
    },
    "seed": 42,
}

## Core classes

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer


class Qwen3Embedder:
    def __init__(
        self,
        model_name: str = "Qwen/Qwen3-Embedding-0.6B",
        device: str = "cuda",
        batch_size: int = 4,
        fp16: bool = True,
    ) -> None:
        model_kwargs = {"torch_dtype": torch.float16} if fp16 else {}
        self.model = SentenceTransformer(
            model_name, device=device, trust_remote_code=True, model_kwargs=model_kwargs
        )
        self.batch_size = batch_size
        self.dim: int = self.model.get_embedding_dimension()

    def encode(self, texts: list) -> np.ndarray:
        vecs = self.model.encode(
            texts,
            batch_size=self.batch_size,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        return np.array(vecs, dtype=np.float32)

    def encode_query(self, texts: list, instruction: str = "") -> np.ndarray:
        if instruction:
            prompt = f"Instruct: {instruction}\nQuery: "
            vecs = self.model.encode(
                texts,
                prompt=prompt,
                batch_size=self.batch_size,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        else:
            vecs = self.model.encode(
                texts,
                batch_size=self.batch_size,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        return np.array(vecs, dtype=np.float32)

In [ ]:
import openai


class OpenAICompatClient:
    def __init__(self, base_url, api_key, model, temperature=0.0, max_completion_tokens=16):
        self.client = openai.OpenAI(base_url=base_url, api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_completion_tokens = max_completion_tokens

    def chat(self, messages: list) -> str:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            max_completion_tokens=self.max_completion_tokens,
        )
        return resp.choices[0].message.content or ""

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    id: str
    split: str
    rendered: str


PERSONAS = {
    "crammer": Profile(
        id="crammer",
        split="train",
        rendered=(
            "A ninth-grader who finds the textbook hard to follow and has little background "
            "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
            "to score — but it must be spelled out simply, step by step, with examples."
        ),
    ),
    "scholar": Profile(
        id="scholar",
        split="train",
        rendered=(
            "A ninth-grader who reads dense material easily and has solid background on this "
            "topic. Wants to understand the underlying why and how, and the connections between "
            "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
        ),
    ),
    "steady": Profile(
        id="steady",
        split="train",
        rendered=(
            "A capable ninth-grader with average background on this topic. Wants a correct "
            "answer with a brief justification, balanced toward exam needs. Does not need "
            "elaborate scaffolding, but does appreciate a one-line reason."
        ),
    ),
}


def render_profile(persona_id: str) -> str:
    return PERSONAS[persona_id].rendered


def train_personas() -> list:
    return [p for p in PERSONAS.values() if p.split == "train"]

## Helper functions

In [ ]:
import json
import logging
import re
from pathlib import Path

from tqdm.auto import tqdm

logging.basicConfig(
    force=True,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

FLOAT_RE = re.compile(r"[\d.]+")

JUDGE_SYSTEM = "You are an expert Persian language tutor evaluating study materials."


def load_corpus(path: Path):
    chunk_ids, texts = [], []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        chunk_ids.append(rec["chunk_id"])
        texts.append(rec["text"])
    if not texts:
        raise ValueError(f"Corpus file {path} is empty")
    return chunk_ids, texts


def parse_float_score(response: str) -> float:
    m = FLOAT_RE.search(response)
    if m is None:
        logger.warning("Could not parse score from: %r", response)
        return 0.0
    return max(0.0, min(1.0, float(m.group())))


def build_judge_messages(query: str, persona_rendered: str, chunk_text: str):
    user = (
        "A student with the following profile is trying to answer an exam question:\n"
        f"Profile: {persona_rendered}\n\n"
        f"Exam question: {query}\n\n"
        f"Candidate study passage:\n{chunk_text}\n\n"
        "Rate 0.0–1.0 how useful this passage is for helping this specific student answer "
        "the question. Consider:\n"
        "  - Does the depth match the student's comprehension level?\n"
        "  - Does it provide what this student needs (simple paraphrase vs. deep analysis)?\n"
        "  - Is the style appropriate (hand-holding vs. terse treatment)?\n\n"
        "Respond with a single decimal number only, e.g. 0.73"
    )
    return [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}]


def score_one_chunk(judge, query, persona_rendered, cid, ctext):
    msgs = build_judge_messages(query, persona_rendered, ctext)
    try:
        score = parse_float_score(judge.chat(msgs))
    except Exception:
        logger.warning("Judge call failed for chunk %s", cid)
        score = 0.0
    return {"chunk_id": cid, "text": ctext, "teacher_score": score}


def score_chunks(judge, query, persona_rendered, chunk_ids, chunk_texts, max_workers=8):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_cid = {
            executor.submit(score_one_chunk, judge, query, persona_rendered, cid, ctext): cid
            for cid, ctext in zip(chunk_ids, chunk_texts, strict=True)
        }
        for future in as_completed(future_to_cid):
            doc = future.result()
            results[doc["chunk_id"]] = doc
    return [results[cid] for cid in chunk_ids]


def load_split_qids(split_path: Path):
    entries = []
    for raw in split_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        exam_stem, qid = line.split(":", 1)
        entries.append((exam_stem.strip(), qid.strip(), line))
    return entries


def load_question(exam_stem: str, qid: str, questions_dir: Path) -> str:
    qfile = questions_dir / f"{exam_stem}.json"
    data = json.loads(qfile.read_text(encoding="utf-8"))
    for q in data.get("questions", []):
        if q["id"] == qid:
            return q["stem"]
    raise KeyError(f"Question {qid!r} not found in {qfile}")


def retrieve_top_k(query_vec, chunk_matrix, top_k, chunk_ids, chunk_texts):
    sims = query_vec.squeeze() @ chunk_matrix.T
    top_idx = np.argsort(sims)[-top_k:][::-1]
    return [chunk_ids[i] for i in top_idx], [chunk_texts[i] for i in top_idx]

## Run pipeline

In [ ]:
np.random.seed(CFG["seed"])

corpus_path = Path(CFG["data"]["chunks"])
questions_dir = Path(CFG["data"]["questions_dir"])
splits_dir = Path(CFG["data"]["splits_dir"])
output_dir = Path(CFG["data"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

logger.info("Loading corpus from %s", corpus_path)
chunk_ids, chunk_texts = load_corpus(corpus_path)
logger.info("%d chunks loaded", len(chunk_ids))

In [ ]:
logger.info("Encoding corpus with %s on %s …", CFG["embedder"]["model"], CFG["embedder"]["device"])
embedder = Qwen3Embedder(
    model_name=CFG["embedder"]["model"],
    device=CFG["embedder"]["device"],
    batch_size=CFG["embedder"]["batch_size"],
    fp16=CFG["embedder"]["fp16"],
)
chunk_matrix = embedder.encode(chunk_texts)
logger.info("Corpus matrix shape: %s", chunk_matrix.shape)

In [ ]:
judge = OpenAICompatClient(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    model=CFG["judge"]["model"],
    temperature=CFG["judge"]["temperature"],
    max_completion_tokens=CFG["judge"]["max_completion_tokens"],
)

top_k = CFG["retriever"]["top_k"]
train_profiles = train_personas()

for split in ("train", "val"):
    split_path = splits_dir / f"{split}_qids.txt"
    if not split_path.exists():
        logger.warning("Split file not found: %s — skipping", split_path)
        continue

    output_path = output_dir / f"{split}.jsonl"
    entries = load_split_qids(split_path)
    logger.info("Processing %s split (%d questions) → %s", split, len(entries), output_path)

    # Pre-load all valid question stems
    valid_entries = []
    for exam_stem, qid, raw_line in entries:
        try:
            query = load_question(exam_stem, qid, questions_dir)
            valid_entries.append((exam_stem, qid, raw_line, query))
        except (FileNotFoundError, KeyError) as exc:
            logger.warning("Skipping %s: %s", raw_line, exc)

    with output_path.open("w", encoding="utf-8") as fh:
        for persona in tqdm(train_profiles, desc=f"{split} personas", unit="p"):
            persona_rendered = render_profile(persona.id)
            query_vecs = embedder.encode_query(
                [e[3] for e in valid_entries], instruction=persona_rendered
            )
            for i, (_, _, raw_line, query) in enumerate(valid_entries):
                query_vec = query_vecs[i : i + 1]
                top_ids, top_texts = retrieve_top_k(
                    query_vec, chunk_matrix, top_k, chunk_ids, chunk_texts
                )
                docs = score_chunks(
                    judge, query, persona_rendered, top_ids, top_texts, CFG["judge"]["max_workers"]
                )
                rec = {"query": query, "persona_id": persona.id, "docs": docs}
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

    logger.info("Wrote %s", output_path)

logger.info("All splits complete. Output in %s", output_dir)

In [ ]:
# ── Derive hard-negative triplets and pairs from the scored data ──────────────
# The original data is listwise-scored: each (query, persona) group has docs
# ranked by an LLM judge (teacher_score in [0,1]). Triplets are derived here by
# treating the highest-scored doc as the positive and the lowest-scored docs as
# negatives — a coarse binarisation of the continuous signal that lets the same
# dataset also drive MNRL (hard_neg mode) training without a separate annotation
# pass. Pairs are a cartesian expansion (one line per negative).
import json
from pathlib import Path

MAX_NEGATIVES = 4

def _derive_triplets(scored_path, output_dir, max_negatives=MAX_NEGATIVES):
    stem = Path(scored_path).stem
    triplet_path = Path(output_dir) / f"{stem}_triplets.jsonl"
    pairs_path   = Path(output_dir) / f"{stem}_pairs.jsonl"
    n_triplets = n_pairs = 0
    with (
        open(triplet_path, "w", encoding="utf-8") as tf,
        open(pairs_path,   "w", encoding="utf-8") as pf,
    ):
        for raw in Path(scored_path).read_text(encoding="utf-8").splitlines():
            if not raw.strip():
                continue
            rec = json.loads(raw)
            docs = rec.get("docs", [])
            if len(docs) < 2:
                continue
            sorted_docs = sorted(docs, key=lambda d: d["teacher_score"], reverse=True)
            positive  = sorted_docs[0]["text"]
            negatives = [d["text"] for d in sorted_docs[-max_negatives:]]
            base = {"query": rec["query"], "persona_id": rec.get("persona_id", "")}
            tf.write(json.dumps({**base, "positive": positive, "negatives": negatives}, ensure_ascii=False) + "\n")
            n_triplets += 1
            for neg in negatives:
                pf.write(json.dumps({**base, "positive": positive, "negative": neg}, ensure_ascii=False) + "\n")
                n_pairs += 1
    print(f"  {Path(scored_path).name} → {triplet_path.name} ({n_triplets} triplets), {pairs_path.name} ({n_pairs} pairs)")

for split in ("train", "val"):
    scored_path = Path(OUTPUT_DIR) / f"{split}.jsonl"
    if scored_path.exists():
        _derive_triplets(scored_path, OUTPUT_DIR)
    else:
        print(f"  {scored_path} not found — skipping")

print("Triplet derivation complete.")